In [2]:
!pip install optuna

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached mako-1.3.10-py3-none-any.whl.metadata (2.9 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 29.8 MB/s  0:00:00
Using cached mako-1.3.10-py3-none-any.whl (78 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)

   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ---------------------------------------- 0/7 [tqdm]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- ---------------------------------- 1/7 [Mako]
   ----- -------------------

In [3]:
# Step 1: Import required libraries
import optuna
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

In [4]:
# Step 2: Load dataset
X, y = load_breast_cancer(return_X_y=True)

# Step 3: Define objective function
# Optuna will try different hyperparameters to MINIMIZE this value
def objective(trial):

    # Step 4: Define hyperparameters search space
    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    max_iter = trial.suggest_int("max_iter", 100, 500)

    # Step 5: Create model using suggested hyperparameters
    model = LogisticRegression(
        C=C,
        max_iter=max_iter,
        solver="liblinear"
    )

    # Step 6: Evaluate model using cross-validation
    score = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    ).mean()

    # Step 7: Optuna minimizes objective, so return negative accuracy
    return -score



In [5]:
# Step 8: Create study (Bayesian optimization)
study = optuna.create_study(direction="minimize")

[I 2026-01-21 20:30:19,981] A new study created in memory with name: no-name-468ac0e0-84b3-436f-a5d4-c36e4b845d51


In [6]:
# Step 9: Start optimization
study.optimize(objective, n_trials=30)

# Step 10: Print best results
print("Best hyperparameters:")
print(study.best_params)

print("\nBest accuracy:")
print(-study.best_value)

[I 2026-01-21 20:30:25,958] Trial 0 finished with value: -0.9525694767893185 and parameters: {'C': 4.581701767651639, 'max_iter': 257}. Best is trial 0 with value: -0.9525694767893185.
[I 2026-01-21 20:30:26,002] Trial 1 finished with value: -0.9261760596180718 and parameters: {'C': 0.0014025369798857407, 'max_iter': 299}. Best is trial 0 with value: -0.9525694767893185.
[I 2026-01-21 20:30:26,042] Trial 2 finished with value: -0.9332246545567457 and parameters: {'C': 0.0354916551231138, 'max_iter': 187}. Best is trial 0 with value: -0.9525694767893185.
[I 2026-01-21 20:30:26,094] Trial 3 finished with value: -0.9525694767893185 and parameters: {'C': 1.131102123635118, 'max_iter': 304}. Best is trial 0 with value: -0.9525694767893185.
[I 2026-01-21 20:30:26,135] Trial 4 finished with value: -0.9402577239559073 and parameters: {'C': 0.06583017266810266, 'max_iter': 255}. Best is trial 0 with value: -0.9525694767893185.
[I 2026-01-21 20:30:26,193] Trial 5 finished with value: -0.95607824

Best hyperparameters:
{'C': 5.587467364039539, 'max_iter': 155}

Best accuracy:
0.956078248719143


In [7]:
# Now use this hyperparameter on your model.

best_model = LogisticRegression(
    C=study.best_params["C"],
    max_iter=study.best_params["max_iter"],
    solver="liblinear"
)

best_model.fit(X, y)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",5.587467364039539
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- 